# 전체 USPTO 특허 Feature 구축

GNN 학습을 위한 **전체 utility 특허 (1976~2016)** 대상 피처 추출.  
참조: `20260609 특허소송예측 데이터구축.ipynb`

| 항목 | 내용 |
|------|------|
| 대상 | PatentsView utility 비취하 특허, grant_date ≤ 2016-12-31 |
| 피처 | MCD, SCD, TD_A~H, ABS, CTS, PTS, CTC, PK, SK, PPS, CD, IC, IE, FIE, ET |
| 출력 | `./data_gnn/patent_features.parquet`, `./data_gnn/patent_company_map.parquet` |
| 데이터 | PatentsView TSV + PACER CSV: `../src/` |

**LI1~LI7 제외**: GNN이 citation+company 그래프 구조에서 소송 전파 패턴을 직접 학습.  
**TCP 제외**: 소송 특허에만 원고 기반 값이 존재해 label leakage 위험.  
**CS 제외**: g_patent_abroad.tsv 미보유, PCT proxy 커버리지 낮아 제외.

---
**처리 순서**
1. PACER → 소송 특허 집합 (`is_litigated` 플래그용)
2. g_patent.tsv → utility 비취하 특허 (~2016)
3. g_assignee_disambiguated.tsv → Patent-Company 매핑, CD, IC
4. 개별 TSV → PK, SK, PPS, ET, IE, FIE, ABS, IPC(MCD/SCD/TD_A~H)
5. g_claims_*.tsv → CTS, PTS, CTC
6. 통합 & 저장

In [ ]:
# Cell 1
import pandas as pd
import numpy as np
import re, os, gc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

os.nice(10)
os.environ['OMP_NUM_THREADS']  = '2'
os.environ['MKL_NUM_THREADS']  = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', 500)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

SRC_DIR  = '../src'
DATA_DIR = './data_gnn'
os.makedirs(DATA_DIR, exist_ok=True)

CHUNKSIZE = 300_000

PATENT_FEAT_COLS = [
    'MCD', 'SCD',
    'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
    'ABS', 'CTS', 'CTC', 'PTS',
    'PK', 'SK', 'PPS',
    'CD', 'IC', 'IE', 'FIE', 'ET',
]

print('환경 설정 완료')
print(f'피처 수: {len(PATENT_FEAT_COLS)}개')
print(f'CHUNKSIZE: {CHUNKSIZE:,}')
print()
print('★ 무거운 순서: Cell7(PK) >> Cell10(IE/FIE) >> Cell12(IPC) >> Cell14(Claims)')

---
## Step 1. 소송 특허 집합 추출 (is_litigated 플래그용)

In [ ]:
# Cell 2
cases   = pd.read_csv(f'{SRC_DIR}/cases.csv',   low_memory=False)
patents = pd.read_csv(f'{SRC_DIR}/patents.csv',  low_memory=False)

inf_cases = cases[cases['case_type_1'] == 1.0]

lit_patents_df = patents[
    (patents['case_row_id'].isin(inf_cases['case_row_id'])) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()

def normalize_patent(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    return m.group(1) + m.group(2).lstrip('0')

lit_patents_df['patent_norm'] = lit_patents_df['patent'].astype(str).apply(normalize_patent)
litigated_set_int = {int(p) for p in lit_patents_df['patent_norm'].dropna() if p.isdigit()}

print(f'침해소송 케이스: {len(inf_cases):,}건')
print(f'소송 특허 (정수 변환): {len(litigated_set_int):,}건')

---
## Step 2. Company 정규화 함수

In [ ]:
# Cell 3
# deep_normalize: 20260625 NPE 정확매칭 노트북 기준
_SUFFIX = re.compile(
    r',?\s*(incorporated|corporation|company|limited|enterprises|'
    r'inc|corp|llc|ltd|co|plc|lp|llp|gmbh|ag|sa|sl|srl|bv|nv|ab|as|kk|'
    r'the|terminated.*|dissolved.*|successor.*)[,.]?\s*$',
    re.IGNORECASE
)
_PREFIX = re.compile(r'^(the)\s+', re.IGNORECASE)

def deep_normalize(name):
    if pd.isna(name):
        return None
    s = str(name).strip().replace('&', 'and')
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip().upper()
    for _ in range(6):
        new = _SUFFIX.sub('', s, count=1).strip()
        if new == s:
            break
        s = new
    return _PREFIX.sub('', s).strip() or None

print('deep_normalize 준비 완료')

---
## Step 3. 전체 Utility 특허 목록

In [ ]:
# Cell 4
print('g_patent.tsv 로딩...', flush=True)
pat_base = pd.read_csv(
    f'{SRC_DIR}/g_patent.tsv', sep='\t',
    usecols=['patent_id', 'patent_type', 'patent_date', 'num_claims', 'withdrawn'],
    low_memory=False
)
pat_base['patent_id'] = pd.to_numeric(pat_base['patent_id'], errors='coerce')
pat_base = pat_base.dropna(subset=['patent_id']).astype({'patent_id': int})

pat_util = pat_base[
    (pat_base['patent_type'] == 'utility') &
    (pat_base['withdrawn'] == 0) &
    (pat_base['patent_date'] <= '2016-12-31')   # 소송 데이터 관측 기간 기준
].copy()
pat_util['grant_date'] = pd.to_datetime(pat_util['patent_date'], errors='coerce')

all_pids = set(pat_util['patent_id'])

print(f'전체 PatentsView 특허:  {len(pat_base):,}건')
print(f'  utility + 비취하 + ~2016: {len(pat_util):,}건')
print(f'  소송 특허 포함:           {len(all_pids & litigated_set_int):,}건')

---
## Step 4. Patent-Company 매핑 + CD / IC 계산

In [ ]:
# Cell 5
print('g_assignee_disambiguated.tsv 로딩...', flush=True)
asgn_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_assignee_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'assignee_sequence', 'disambig_assignee_organization', 'location_id'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        asgn_chunks.append(chunk)

asgn_df = pd.concat(asgn_chunks, ignore_index=True)
asgn_df['org_norm'] = asgn_df['disambig_assignee_organization'].apply(deep_normalize)

# CD: 특허당 출원인 수
cd_df = (
    asgn_df.groupby('patent_id')['assignee_sequence'].count()
    .reset_index().rename(columns={'assignee_sequence': 'CD'})
)

# IC: 특허당 출원인 국가 수 (location_id → raw_country)
need_locs = set(asgn_df['location_id'].dropna().unique())
loc_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_location_not_disambiguated.tsv', sep='\t',
    usecols=['location_id', 'raw_country'],
    low_memory=False, chunksize=CHUNKSIZE
):
    c = chunk[chunk['location_id'].isin(need_locs)]
    if len(c) > 0:
        loc_chunks.append(c)
loc_df = pd.concat(loc_chunks).drop_duplicates('location_id')[['location_id', 'raw_country']]
ic_df = (
    asgn_df.merge(loc_df, on='location_id', how='left')
    .groupby('patent_id')['raw_country'].nunique()
    .reset_index().rename(columns={'raw_country': 'IC'})
)

# 특허당 대표 출원인 (assignee_sequence 최소 = 주 출원인)
patent_company = (
    asgn_df.sort_values('assignee_sequence')
    .dropna(subset=['org_norm'])
    .drop_duplicates('patent_id')
    [['patent_id', 'org_norm']]
)

print(f'Assignee 로딩 완료: {len(asgn_df):,}행')
print(f'CD: {len(cd_df):,}건 / IC: {len(ic_df):,}건')
print(f'Patent-Company 매핑: {len(patent_company):,}건')
del asgn_chunks; gc.collect()

In [ ]:
# Cell 6-ckpt  ★ 커널 재시작 후 여기서 체크포인트 로드 (이미 완료된 셀 건너뜀)
import os

def load_ckpt(path, name):
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'  ✓ {name} 로드 ({len(df):,}건) ← {path}')
        return df
    else:
        print(f'  ✗ {name} 체크포인트 없음 → 해당 셀 직접 실행 필요')
        return None

print('=== 체크포인트 현황 ===')
if 'pk_df' not in dir():
    pk_df = load_ckpt(f'{DATA_DIR}/ckpt_pk.parquet', 'PK')
if 'sk_df' not in dir():
    _tmp = load_ckpt(f'{DATA_DIR}/ckpt_sk_pps.parquet', 'SK+PPS')
    if _tmp is not None:
        sk_df  = _tmp[['patent_id','SK']].dropna(subset=['SK'])
        pps_df = _tmp[['patent_id','PPS']].dropna(subset=['PPS'])
if 'et_df' not in dir():
    et_df = load_ckpt(f'{DATA_DIR}/ckpt_et.parquet', 'ET')
if 'ie_df' not in dir():
    _tmp = load_ckpt(f'{DATA_DIR}/ckpt_ie_fie.parquet', 'IE+FIE')
    if _tmp is not None:
        ie_df  = _tmp[['patent_id','IE']].dropna(subset=['IE'])
        fie_df = _tmp[['patent_id','FIE']].dropna(subset=['FIE'])
if 'abs_df' not in dir():
    abs_df = load_ckpt(f'{DATA_DIR}/ckpt_abs.parquet', 'ABS')
if 'ipc_feat' not in dir():
    ipc_feat = load_ckpt(f'{DATA_DIR}/ckpt_ipc.parquet', 'IPC')
if 'claims_feat' not in dir():
    claims_feat = load_ckpt(f'{DATA_DIR}/ckpt_claims.parquet', 'Claims')

---
## Step 5. PatentsView 개별 피처 추출

In [ ]:
# Cell 7
# PK: 인용한 선행기술 수 (outgoing citations) — 대용량 chunked
# ★ 가장 오래 걸리는 단계 (수 시간 소요 가능)
print('PK 추출 중... (대용량 파일, 시간 소요)', flush=True)
pk_acc = {}
for i, chunk in enumerate(pd.read_csv(
    f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t',
    usecols=['patent_id', 'citation_sequence'],
    low_memory=False, chunksize=CHUNKSIZE
)):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    for pid, cnt in chunk.groupby('patent_id').size().items():
        pk_acc[int(pid)] = pk_acc.get(int(pid), 0) + int(cnt)
    if i % 20 == 0:
        print(f'  chunk {i:3d}: {len(pk_acc):,}건 누적', flush=True)

pk_df = pd.DataFrame(list(pk_acc.items()), columns=['patent_id', 'PK'])
pk_df.to_parquet(f'{DATA_DIR}/ckpt_pk.parquet', index=False)
print(f'PK 완료: {len(pk_df):,}건  →  ckpt_pk.parquet 저장')
del pk_acc; gc.collect()

In [ ]:
# Cell 8
# SK: 비특허 선행기술 인용 수
print('SK 추출...', flush=True)
sk_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_other_reference.tsv', sep='\t',
    usecols=['patent_id', 'other_reference_sequence'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        sk_chunks.append(chunk)
sk_df = (
    pd.concat(sk_chunks).groupby('patent_id').size()
    .reset_index().rename(columns={0: 'SK'})
)
print(f'SK: {len(sk_df):,}건')
del sk_chunks; gc.collect()

# PPS: 외국 우선권 국가 수
print('PPS 추출...', flush=True)
pps_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_foreign_priority.tsv', sep='\t',
    usecols=['patent_id', 'foreign_country_filed'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        pps_chunks.append(chunk)
pps_df = (
    pd.concat(pps_chunks).groupby('patent_id')['foreign_country_filed'].nunique()
    .reset_index().rename(columns={'foreign_country_filed': 'PPS'})
)
print(f'PPS: {len(pps_df):,}건')
del pps_chunks; gc.collect()

pd.merge(sk_df, pps_df, on='patent_id', how='outer').to_parquet(f'{DATA_DIR}/ckpt_sk_pps.parquet', index=False)
print('ckpt_sk_pps.parquet 저장')

In [ ]:
# Cell 9
# ET: 심사 기간 (grant_date - filing_date, 일)
print('ET 추출...', flush=True)
app_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_application.tsv', sep='\t',
    usecols=['patent_id', 'filing_date'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        app_chunks.append(chunk)

app_df = pd.concat(app_chunks).drop_duplicates('patent_id')
et_df = pat_util[['patent_id', 'grant_date']].merge(app_df, on='patent_id', how='inner')

filing_dt = pd.to_datetime(et_df['filing_date'], errors='coerce')
filing_dt = filing_dt.where(filing_dt >= '1970-01-01', other=pd.NaT)  # 소송 특허 최초 출원 1973년 기준

et_df['ET'] = (et_df['grant_date'] - filing_dt).dt.days
et_df = et_df[['patent_id', 'ET']].dropna()
et_df.to_parquet(f'{DATA_DIR}/ckpt_et.parquet', index=False)
print(f'ET: {len(et_df):,}건  →  ckpt_et.parquet 저장')
del app_chunks; gc.collect()

In [ ]:
# Cell 10
# IE: 발명자 수 / FIE: 발명자 국가 수
print('IE / FIE 추출...', flush=True)
inv_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_inventor_not_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'inventor_sequence', 'rawlocation_id'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        inv_chunks.append(chunk)

inv_df = pd.concat(inv_chunks, ignore_index=True)
ie_df = (
    inv_df.groupby('patent_id')['inventor_sequence'].count()
    .reset_index().rename(columns={'inventor_sequence': 'IE'})
)

need_locs_inv = set(inv_df['rawlocation_id'].dropna().unique())
loc2_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_location_not_disambiguated.tsv', sep='\t',
    usecols=['rawlocation_id', 'raw_country'],
    low_memory=False, chunksize=CHUNKSIZE
):
    c = chunk[chunk['rawlocation_id'].isin(need_locs_inv)]
    if len(c) > 0:
        loc2_chunks.append(c)
loc2_df = pd.concat(loc2_chunks).drop_duplicates('rawlocation_id')

fie_df = (
    inv_df.merge(loc2_df, on='rawlocation_id', how='left')
    .groupby('patent_id')['raw_country'].nunique()
    .reset_index().rename(columns={'raw_country': 'FIE'})
)
print(f'IE: {len(ie_df):,}건 / FIE: {len(fie_df):,}건')
del inv_chunks, inv_df; gc.collect()

pd.merge(ie_df, fie_df, on='patent_id', how='outer').to_parquet(f'{DATA_DIR}/ckpt_ie_fie.parquet', index=False)
print('ckpt_ie_fie.parquet 저장')

In [ ]:
# Cell 11
# ABS: 초록 단어 수
print('ABS 추출...', flush=True)
abs_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_patent_abstract.tsv', sep='\t',
    usecols=['patent_id', 'patent_abstract'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        abs_chunks.append(chunk)

abs_raw = pd.concat(abs_chunks)
abs_raw['ABS'] = abs_raw['patent_abstract'].fillna('').apply(
    lambda x: len(str(x).split())
)
abs_df = abs_raw[['patent_id', 'ABS']].drop_duplicates('patent_id')
abs_df.to_parquet(f'{DATA_DIR}/ckpt_abs.parquet', index=False)
print(f'ABS: {len(abs_df):,}건  →  ckpt_abs.parquet 저장')
del abs_chunks, abs_raw; gc.collect()

In [ ]:
# Cell 12
# IPC: MCD(주분류 수), SCD(부분류 수), TD_A~H(섹션 유무)
print('IPC 추출...', flush=True)
ipc_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_ipc_at_issue.tsv', sep='\t',
    usecols=['patent_id', 'section', 'ipc_class', 'subclass'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        ipc_chunks.append(chunk)

ipc_df = pd.concat(ipc_chunks, ignore_index=True)
ipc_df['main_class'] = (
    ipc_df['section'].fillna('').astype(str) +
    ipc_df['ipc_class'].fillna(0).astype(str)
)
ipc_df['sub_class'] = ipc_df['main_class'] + ipc_df['subclass'].fillna('').astype(str)

ipc_rows = []
for pid, grp in ipc_df.groupby('patent_id'):
    row = {
        'patent_id': pid,
        'MCD': grp['main_class'].nunique(),
        'SCD': grp['sub_class'].nunique()
    }
    secs = set(grp['section'].dropna().astype(str).str.upper())
    for s in 'ABCDEFGH':
        row[f'TD_{s}'] = int(s in secs)
    ipc_rows.append(row)

ipc_feat = pd.DataFrame(ipc_rows)
ipc_feat.to_parquet(f'{DATA_DIR}/ckpt_ipc.parquet', index=False)
print(f'IPC: {len(ipc_feat):,}건  →  ckpt_ipc.parquet 저장')
del ipc_chunks, ipc_df; gc.collect()

---
## Step 6. Claims 피처 (CTS / PTS / CTC)

`g_claims_YYYY.tsv` 파일이 `../src/` 에 있으면 연도별로 처리.  
없으면 `num_claims` (g_patent.tsv)로 CTS를 대체하고 PTS / CTC = NaN.

In [ ]:
# Cell 14 (Claims) — patentsview_data/ 에 있는 TSV 전부 처리
# 사용법: TSV 올리고 실행 → 처리완료 → TSV 삭제 → 다음 배치 올리고 반복
CLAIMS_DIR = Path('./patentsview_data')
CLAIMS_CKPT_DIR = Path(DATA_DIR) / 'claims_by_year'
CLAIMS_CKPT_DIR.mkdir(exist_ok=True)

tsv_files = sorted(CLAIMS_DIR.glob('g_claims_*.tsv'))
print(f'발견된 TSV: {len(tsv_files)}개 → {[f.name for f in tsv_files]}')

for cfile in tsv_files:
    year = cfile.stem.split('_')[-1]
    ckpt_path = CLAIMS_CKPT_DIR / f'claims_{year}.parquet'

    if ckpt_path.exists():
        print(f'  {year}: 이미 처리됨 (스킵)')
        continue

    print(f'  {year} 처리 중...', flush=True)
    df = pd.read_csv(cfile, sep='\t', usecols=['patent_id', 'claim_text', 'dependent'], low_memory=False)
    df['patent_id'] = pd.to_numeric(df['patent_id'], errors='coerce')
    df = df[df['patent_id'].isin(all_pids)].copy()

    if len(df) == 0:
        print(f'  {year}: 매칭 특허 0건')
    else:
        df['wc'] = df['claim_text'].fillna('').str.split().str.len()
        year_feat = pd.concat([
            df.groupby('patent_id').size().rename('CTS'),
            df[df['dependent'].isna()].groupby('patent_id').size().rename('PTS'),
            df.groupby('patent_id')['wc'].sum().rename('CTC'),
        ], axis=1).reset_index()
        year_feat.to_parquet(ckpt_path, index=False)
        print(f'  {year} 완료: {len(year_feat):,}건 → claims_{year}.parquet 저장')
        del df, year_feat; gc.collect()

year_ckpts = sorted(CLAIMS_CKPT_DIR.glob('claims_*.parquet'))
done_years = [p.stem.replace('claims_', '') for p in year_ckpts]
print(f'\n완료된 연도 ({len(done_years)}개): {done_years}')

In [ ]:
# Cell 14-combine — 모든 연도 완료 후 한 번만 실행
year_ckpts = sorted(CLAIMS_CKPT_DIR.glob('claims_*.parquet'))
print(f'합치기: {len(year_ckpts)}개 연도')

if year_ckpts:
    claims_all = pd.concat([pd.read_parquet(p) for p in year_ckpts], ignore_index=True)
    claims_feat = (
        claims_all.groupby('patent_id')
        .agg({'CTS': 'sum', 'PTS': 'sum', 'CTC': 'sum'})
        .reset_index()
    )
    claims_feat.to_parquet(f'{DATA_DIR}/ckpt_claims.parquet', index=False)
    print(f'Claims 통합 완료: {len(claims_feat):,}건  →  ckpt_claims.parquet 저장')
    del claims_all; gc.collect()
else:
    print('처리된 연도 없음 → num_claims로 CTS 대체')
    claims_feat = pat_util[['patent_id', 'num_claims']].rename(columns={'num_claims': 'CTS'}).copy()
    claims_feat['PTS'] = np.nan
    claims_feat['CTC'] = np.nan

---
## Step 7. 특허 피처 통합 + 소송 플래그

In [ ]:
# Cell 15
print('특허 피처 통합 중...', flush=True)

feat = pat_util[['patent_id', 'grant_date']].copy()

for df_, name in [
    (pk_df,       'PK'),
    (sk_df,       'SK'),
    (pps_df,      'PPS'),
    (et_df,       'ET'),
    (ie_df,       'IE'),
    (fie_df,      'FIE'),
    (abs_df,      'ABS'),
    (cd_df,       'CD'),
    (ic_df,       'IC'),
    (ipc_feat,    'IPC'),
    (claims_feat, 'Claims'),
]:
    feat = feat.merge(df_, on='patent_id', how='left')
    print(f'  + {name:8s} → {len(feat):,}건')

feat = feat.merge(patent_company, on='patent_id', how='left')
feat['is_litigated'] = feat['patent_id'].isin(litigated_set_int).astype(np.int8)

print(f'\n통합 완료: {len(feat):,}건 / {len(feat.columns)}컬럼')
print(f'소송: {feat["is_litigated"].sum():,}건 / 비소송: {(feat["is_litigated"]==0).sum():,}건')

---
## Step 8. 결측 처리

In [ ]:
# Cell 16
FILL_ZERO   = ['PK', 'SK', 'PPS', 'MCD', 'SCD',
               'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
               'CTS', 'PTS', 'CTC']
FILL_MEDIAN = ['ABS', 'IE', 'FIE', 'ET', 'CD', 'IC']

for col in FILL_ZERO:
    if col in feat.columns:
        feat[col] = feat[col].fillna(0)
for col in FILL_MEDIAN:
    if col in feat.columns:
        feat[col] = feat[col].fillna(feat[col].median())

remaining_na = feat[PATENT_FEAT_COLS].isna().sum()
print('결측 처리 완료 — 잔여 결측:')
print(remaining_na[remaining_na > 0] if remaining_na.any() else '  없음')

---
## Step 9. 저장

In [ ]:
# Cell 17
save_cols = ['patent_id', 'org_norm', 'grant_date', 'is_litigated'] + PATENT_FEAT_COLS
save_cols = [c for c in save_cols if c in feat.columns]

feat[save_cols].to_parquet(f'{DATA_DIR}/patent_features.parquet', index=False)
print(f'patent_features.parquet: {len(feat):,}건 / {len(save_cols)}컬럼')

feat[['patent_id', 'org_norm']].to_parquet(f'{DATA_DIR}/patent_company_map.parquet', index=False)
print(f'patent_company_map.parquet: {len(feat):,}건')

---
## Step 10. 커버리지 리포트

In [ ]:
# Cell 17
cov_rows = []
for col in PATENT_FEAT_COLS:
    if col not in feat.columns:
        continue
    cov_rows.append({
        'Feature':   col,
        'All':       feat[col].notna().mean(),
        'Litigated': feat.loc[feat['is_litigated']==1, col].notna().mean(),
        'Non-Lit':   feat.loc[feat['is_litigated']==0, col].notna().mean(),
    })

cov_df = pd.DataFrame(cov_rows).set_index('Feature')
print('=== 피처 커버리지 ===')
print(cov_df.round(3).to_string())
cov_df.to_csv(f'{DATA_DIR}/feature_coverage.csv', encoding='utf-8-sig')

In [ ]:
# Cell 18
feat_present = [c for c in PATENT_FEAT_COLS if c in feat.columns]
comp = feat.groupby('is_litigated')[feat_present].mean().T
comp.columns = ['Non-Lit (0)', 'Litigated (1)']
comp['L/NL ratio'] = (comp['Litigated (1)'] / (comp['Non-Lit (0)'] + 1e-8)).round(2)

print('=== 소송 vs 비소송 피처 평균 비교 ===')
print(comp.sort_values('L/NL ratio', ascending=False).round(3).to_string())

print('\n=== 저장 파일 ===')
for f in sorted(Path(DATA_DIR).glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:45s}  {size_mb:8.1f} MB')

---
## Step 11. ML Baseline — Model 1: L vs NL 소송 예측 (LightGBM + Optuna)

TCP·LI 제거 후 **22개 피처**만으로 소송 예측 성능 검증.  
GNN 추가 전 baseline 확인 목적.

In [ ]:
# Cell 19
import pickle
import lightgbm as lgb
import optuna
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42

def _check_lgb_gpu():
    try:
        m = lgb.LGBMClassifier(device='gpu', n_estimators=5, verbose=-1)
        m.fit(np.random.rand(50, 3), np.random.randint(0, 2, 50))
        return True
    except:
        return False

USE_GPU = _check_lgb_gpu()
DEVICE  = 'gpu' if USE_GPU else 'cpu'
print(f'LightGBM device: {DEVICE}')
print(f'피처 수: {len(PATENT_FEAT_COLS)}개')

In [ ]:
# Cell 20
df_all = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet')
print(f'전체: {len(df_all):,}건  |  L={df_all["is_litigated"].sum():,}  NL={(df_all["is_litigated"]==0).sum():,}')

df_L  = df_all[df_all['is_litigated'] == 1].reset_index(drop=True)
df_NL = df_all[df_all['is_litigated'] == 0].reset_index(drop=True)

# L/NL 각각 8:2 분할
L_train,       L_test       = train_test_split(df_L,  test_size=0.2, random_state=RANDOM_STATE)
NL_train_pool, NL_test      = train_test_split(df_NL, test_size=0.2, random_state=RANDOM_STATE)

# 학습: 1:1 언더샘플링 (기존 모델과 동일)
n_needed = len(L_train)
NL_train = NL_train_pool.sample(n=n_needed, random_state=RANDOM_STATE)

train_m1 = pd.concat([L_train, NL_train]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_m1  = pd.concat([L_test,  NL_test ]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'\n학습셋: {len(train_m1):,}건  (L={len(L_train):,} / NL={len(NL_train):,}, 1:1 균형)')
print(f'테스트셋: {len(test_m1):,}건  (L={len(L_test):,} / NL={len(NL_test):,}, 자연 불균형)')

In [ ]:
# Cell 21
feat_cols = PATENT_FEAT_COLS  # 22개

M1_FEAT_SETS = {
    'all_22':  feat_cols,
    'no_td_d': [f for f in feat_cols if f != 'TD_D'],  # TD_D: ~0.8% non-zero
}

print('=== M1 피처셋 ===')
for name, fs in M1_FEAT_SETS.items():
    print(f'  {name:10s} ({len(fs):2d}개): {fs}')

In [ ]:
# Cell 22
def _lgb_params(trial, extra=None):
    p = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500, step=100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'device': DEVICE, 'n_jobs': 1, 'random_state': RANDOM_STATE, 'verbose': -1,
    }
    if extra:
        p.update(extra)
    return p

def make_objective_m1(feat_set, df_train):
    X = df_train[feat_set].values
    y = df_train['is_litigated'].values
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    def objective(trial):
        model = lgb.LGBMClassifier(**_lgb_params(trial))
        aucs = []
        for tr_idx, val_idx in cv.split(X, y):
            model.fit(X[tr_idx], y[tr_idx])
            proba = model.predict_proba(X[val_idx])[:, 1]
            aucs.append(roc_auc_score(y[val_idx], proba))
        return np.mean(aucs)
    return objective

print('Optuna 목적함수 정의 완료')

In [ ]:
# Cell 23
from sklearn.metrics import matthews_corrcoef
import os

N_TRIALS = 40
TIMEOUT  = 300
M1_CKPT  = f'{DATA_DIR}/m1_results.pkl'

def threshold_metrics(y_true, proba, threshold=None):
    if threshold is None:
        best_f2, best_thr = 0, 0.5
        for thr in np.linspace(0.01, 0.99, 99):
            pred = (proba >= thr).astype(int)
            tp = ((pred==1)&(y_true==1)).sum()
            fp = ((pred==1)&(y_true==0)).sum()
            fn = ((pred==0)&(y_true==1)).sum()
            p = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9)
            f2 = 5*p*r/(4*p+r+1e-9)
            if f2 > best_f2: best_f2, best_thr = f2, thr
        threshold = best_thr
    pred = (proba >= threshold).astype(int)
    tp = ((pred==1)&(y_true==1)).sum(); fp = ((pred==1)&(y_true==0)).sum()
    fn = ((pred==0)&(y_true==1)).sum(); tn = ((pred==0)&(y_true==0)).sum()
    p = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
    return {'threshold': threshold,
            'BA': (r+sp)/2, 'Prec': p, 'Rec': r,
            'F1': 2*p*r/(p+r+1e-9), 'F2': 5*p*r/(4*p+r+1e-9),
            'Gmean': np.sqrt(r*sp), 'MCC': matthews_corrcoef(y_true, pred)}

# ── 모델 로드 or 튜닝 ──────────────────────────────────────────────────────
if os.path.exists(M1_CKPT):
    with open(M1_CKPT, 'rb') as f:
        m1_models = pickle.load(f)   # {set_name: {model, feat_set, best_params, cv_auc}}
    print(f'✓ 모델 로드 ({list(m1_models.keys())})  → predict_proba 재계산')
else:
    m1_models = {}
    for set_name, fs in M1_FEAT_SETS.items():
        print(f'\n[Model1] {set_name} ({len(fs)}개 피처) 튜닝 중...')
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
        )
        study.optimize(make_objective_m1(fs, train_m1),
                       n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)
        best_params = study.best_params.copy()
        best_params.update({'device': DEVICE, 'n_jobs': -1,
                            'random_state': RANDOM_STATE, 'verbose': -1})
        model = lgb.LGBMClassifier(**best_params)
        model.fit(train_m1[fs], train_m1['is_litigated'])
        m1_models[set_name] = {
            'model': model, 'feat_set': fs,
            'best_params': best_params, 'cv_auc': study.best_value,
        }
        print(f'  완료  CV AUC={study.best_value:.4f}')

    with open(M1_CKPT, 'wb') as f:
        pickle.dump(m1_models, f)
    print(f'\n✓ 모델 저장 → {M1_CKPT}')

# ── predict_proba + 지표 계산 (항상 실행) ─────────────────────────────────
m1_results = {}
y_te = test_m1['is_litigated'].values

for set_name, m in m1_models.items():
    fs    = m['feat_set']
    model = m['model']
    proba = model.predict_proba(test_m1[fs])[:, 1]
    tr_proba = model.predict_proba(train_m1[fs])[:, 1]
    best_thr = threshold_metrics(train_m1['is_litigated'].values, tr_proba)['threshold']
    te_m     = threshold_metrics(y_te, proba, threshold=best_thr)
    m1_results[set_name] = {
        **m,
        'test_auc': roc_auc_score(y_te, proba),
        'test_ap':  average_precision_score(y_te, proba),
        'test_proba': proba, 'y_te': y_te, 'threshold': best_thr,
        **{f'test_{k}': v for k, v in te_m.items() if k != 'threshold'},
    }

# ── 비교표 ─────────────────────────────────────────────────────────────────
HDR = f'{"모델":22s}  {"BA":6s}  {"Prec":6s}  {"Rec":6s}  {"F1":6s}  {"F2":6s}  {"Gmean":6s}  {"MCC":6s}'
print('\n' + '='*80)
print('  Model 1 — L vs NL 비교')
print('='*80)
print(HDR); print('-'*80)
# 논문 기존 수치
print(f'{"논문(기존)":22s}  0.700   0.016   0.629   0.031   0.073   0.697   0.073')
print('-'*80)
for name, r in m1_results.items():
    tag = f'LGB-{name}'
    print(f'{tag:22s}  {r["test_BA"]:.4f}  {r["test_Prec"]:.4f}  {r["test_Rec"]:.4f}  '
          f'{r["test_F1"]:.4f}  {r["test_F2"]:.4f}  {r["test_Gmean"]:.4f}  {r["test_MCC"]:.4f}')
print('='*80)
print(f'  AUC / AP: ', '  '.join(
    f'{n}: AUC={r["test_auc"]:.4f} AP={r["test_ap"]:.4f}'
    for n, r in m1_results.items()))

In [ ]:
# Cell 24
best_m1_name = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
best_m1 = m1_results[best_m1_name]
print(f'Best 피처셋: {best_m1_name}  (Test AUC {best_m1["test_auc"]:.4f})')

shap_sample = train_m1[best_m1['feat_set']].sample(min(2000, len(train_m1)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(best_m1['model'])
shap_vals = explainer.shap_values(shap_sample)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

shap_imp = pd.Series(
    np.abs(sv).mean(axis=0),
    index=best_m1['feat_set']
).sort_values(ascending=False)

print('\n=== SHAP Feature Importance ===')
print(shap_imp.round(4).to_string())

In [ ]:
# Cell 25
import datetime
today = datetime.date.today().strftime('%Y%m%d')

best_m1_name = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
best_m1 = m1_results[best_m1_name]

save_m1 = {
    'name':        f'model1_final_{today}',
    'feat_cols':   best_m1['feat_set'],
    'model':       best_m1['model'],
    'best_params': best_m1['best_params'],
    'cv_auc':      best_m1['cv_auc'],
    'test_auc':    best_m1['test_auc'],
    'test_ap':     best_m1['test_ap'],
}
with open(f'{DATA_DIR}/model1_final.pkl', 'wb') as f:
    pickle.dump(save_m1, f)
print(f'모델 저장: {DATA_DIR}/model1_final.pkl')
print(f'  이름: {save_m1["name"]}')
print(f'  피처: {len(save_m1["feat_cols"])}개')
print(f'  Test AUC: {save_m1["test_auc"]:.4f}')
print(f'  Test AP:  {save_m1["test_ap"]:.4f}')

---
## Step 13. GNN Baseline — GraphSAGE + LightGBM

citation 그래프 구조를 활용해 ML baseline 대비 성능 개선 측정.  
전체 5.5M 노드 semi-supervised: message passing은 전체, loss는 train 레이블만.

In [ ]:
# Cell 26 — GNN 환경 설정 + 노드 피처 텐서
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data

GNN_CKPT_DIR = Path(DATA_DIR) / 'gnn'
GNN_CKPT_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}'
          f'  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

# 노드 피처 로드
df_gnn = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet')
df_gnn = df_gnn.reset_index(drop=True)
pid_to_idx = {int(pid): i for i, pid in enumerate(df_gnn['patent_id'].values)}
N = len(df_gnn)

x = torch.tensor(df_gnn[PATENT_FEAT_COLS].values, dtype=torch.float32)
y = torch.tensor(df_gnn['is_litigated'].values,    dtype=torch.long)

# train/test 마스크 — Cell 20의 동일 split 사용
train_pids = set(train_m1['patent_id'].values.tolist())
test_pids  = set(test_m1['patent_id'].values.tolist())

train_mask = torch.zeros(N, dtype=torch.bool)
test_mask  = torch.zeros(N, dtype=torch.bool)
for pid, idx in pid_to_idx.items():
    if pid in train_pids:  train_mask[idx] = True
    if pid in test_pids:   test_mask[idx]  = True

print(f'노드 수: {N:,}  |  피처: {x.shape[1]}')
print(f'train mask: {train_mask.sum():,}  test mask: {test_mask.sum():,}')
L_in_train = y[train_mask].sum().item()
print(f'train L={L_in_train:,} / NL={train_mask.sum().item()-L_in_train:,}')

In [ ]:
# Cell 27 — Citation 엣지 구성 + PyG Data 생성 (체크포인트)
EDGE_CKPT = GNN_CKPT_DIR / 'edge_index.pt'

if EDGE_CKPT.exists():
    edge_index = torch.load(EDGE_CKPT, weights_only=True)
    print(f'✓ edge_index 로드: {edge_index.shape[1]:,}개 엣지')
else:
    # pandas로 컬럼명 읽기 (따옴표 처리 자동)
    cit_cols = pd.read_csv(
        f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t', nrows=0
    ).columns.tolist()
    print('citation 컬럼:', cit_cols)

    cit_id_col = next((c for c in cit_cols
                       if 'citation' in c.lower() and 'patent' in c.lower()
                       and c != 'patent_id'), None)
    if cit_id_col is None:
        cit_id_col = [c for c in cit_cols if c != 'patent_id' and 'id' in c.lower()][0]
    print(f'  citing: patent_id  →  cited: {cit_id_col}')

    pid_series = pd.Series(pid_to_idx)  # pid → node_idx (벡터화 map용)

    src_parts, dst_parts = [], []
    for i, chunk in enumerate(pd.read_csv(
        f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t',
        usecols=['patent_id', cit_id_col], low_memory=False, chunksize=300_000
    )):
        chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
        chunk[cit_id_col]  = pd.to_numeric(chunk[cit_id_col],  errors='coerce')
        chunk = chunk.dropna().astype(int)

        s_idx = chunk['patent_id'].map(pid_series)
        d_idx = chunk[cit_id_col].map(pid_series)
        valid = s_idx.notna() & d_idx.notna()
        s = s_idx[valid].astype(int).values
        d = d_idx[valid].astype(int).values

        src_parts.append(np.concatenate([s, d]))   # 양방향
        dst_parts.append(np.concatenate([d, s]))

        if (i + 1) % 10 == 0:
            print(f'  chunk {i+1:3d}  누적 엣지: {sum(len(x) for x in src_parts):,}')

    src_arr = np.concatenate(src_parts)
    dst_arr = np.concatenate(dst_parts)
    edge_index = torch.tensor([src_arr, dst_arr], dtype=torch.long)
    torch.save(edge_index, EDGE_CKPT)
    print(f'✓ edge_index 저장: {edge_index.shape[1]:,}개 엣지 (양방향)')
    del src_parts, dst_parts, src_arr, dst_arr, pid_series

data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask
data.test_mask  = test_mask
print(f'\nPyG Data: {data}')
print(f'  엣지 밀도: {edge_index.shape[1]/N:.1f} edges/node 평균')

In [ ]:
# Cell 28 — GraphSAGE 모델 + 학습 (체크포인트)
GNN_MODEL_CKPT = GNN_CKPT_DIR / 'graphsage.pt'

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_c, hidden_c=128, emb_c=64):
        super().__init__()
        self.conv1 = SAGEConv(in_c, hidden_c)
        self.conv2 = SAGEConv(hidden_c, emb_c)
        self.head  = torch.nn.Linear(emb_c, 1)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index)

    def forward(self, x, edge_index):
        return self.head(self.encode(x, edge_index)).squeeze(-1)

if GNN_MODEL_CKPT.exists():
    model_gnn = GraphSAGE(in_c=len(PATENT_FEAT_COLS)).to(device)
    model_gnn.load_state_dict(torch.load(GNN_MODEL_CKPT, map_location=device))
    model_gnn.eval()
    print(f'✓ GraphSAGE 로드: {GNN_MODEL_CKPT}')
else:
    # 클래스 불균형 보정
    n_pos = int(y[train_mask].sum())
    n_neg = int(train_mask.sum()) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
    criterion  = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # NeighborLoader — seed: train 노드만, neighbor sampling: [25, 10]
    loader = NeighborLoader(
        data, num_neighbors=[25, 10], batch_size=512,
        input_nodes=data.train_mask, shuffle=True,
    )

    model_gnn = GraphSAGE(in_c=len(PATENT_FEAT_COLS)).to(device)
    optimizer = torch.optim.Adam(model_gnn.parameters(), lr=1e-3, weight_decay=1e-5)

    EPOCHS = 50
    print(f'학습 시작 (epochs={EPOCHS}, device={device})')
    for epoch in range(1, EPOCHS + 1):
        model_gnn.train()
        total_loss, n_batch = 0.0, 0
        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            # batch.n_id[:batch.batch_size] → seed 노드 (train 노드)
            out  = model_gnn(batch.x, batch.edge_index)[:batch.batch_size]
            lbl  = batch.y[:batch.batch_size].float()
            loss = criterion(out, lbl)
            loss.backward()
            optimizer.step()
            total_loss += loss.item(); n_batch += 1
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss/n_batch:.4f}')

    torch.save(model_gnn.state_dict(), GNN_MODEL_CKPT)
    print(f'✓ 모델 저장: {GNN_MODEL_CKPT}')

In [ ]:
# Cell 29 — 임베딩 추출 + LightGBM (GNN 피처 추가) + 비교
EMB_CKPT = GNN_CKPT_DIR / 'embeddings.pt'

if EMB_CKPT.exists():
    emb_all = torch.load(EMB_CKPT, map_location='cpu').numpy()
    print(f'✓ 임베딩 로드: {emb_all.shape}')
else:
    # 전체 노드 임베딩 추출 (NeighborLoader로 배치 추론)
    inf_loader = NeighborLoader(
        data, num_neighbors=[25, 10], batch_size=1024,
        input_nodes=torch.arange(N), shuffle=False,
    )
    model_gnn.eval()
    emb_list = []
    with torch.no_grad():
        for batch in inf_loader:
            batch = batch.to(device)
            emb   = model_gnn.encode(batch.x, batch.edge_index)[:batch.batch_size]
            emb_list.append(emb.cpu())
    emb_all = torch.cat(emb_list, dim=0).numpy()   # (N, 64)
    torch.save(torch.tensor(emb_all), EMB_CKPT)
    print(f'✓ 임베딩 저장: {emb_all.shape}')

# 임베딩 컬럼명
EMB_COLS = [f'gnn_{i}' for i in range(emb_all.shape[1])]
df_emb   = pd.DataFrame(emb_all, columns=EMB_COLS)
df_emb['patent_id'] = df_gnn['patent_id'].values

# 기존 피처 + GNN 임베딩 합치기
df_aug = df_gnn[['patent_id', 'is_litigated'] + PATENT_FEAT_COLS].merge(
    df_emb, on='patent_id', how='left')

FEAT_GNN = PATENT_FEAT_COLS + EMB_COLS   # 22 + 64 = 86개

# 동일 train/test 인덱스로 분할
aug_train = df_aug[df_aug['patent_id'].isin(train_pids)].copy()
aug_test  = df_aug[df_aug['patent_id'].isin(test_pids)].copy()
print(f'aug_train: {len(aug_train):,}  aug_test: {len(aug_test):,}')

# LightGBM — Cell 23 best 파라미터 재사용
best_set  = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
base_params = m1_results[best_set]['best_params'].copy()
base_params.update({'device': DEVICE, 'n_jobs': -1,
                    'random_state': RANDOM_STATE, 'verbose': -1})
lgb_gnn = lgb.LGBMClassifier(**base_params)
lgb_gnn.fit(aug_train[FEAT_GNN], aug_train['is_litigated'])

y_te     = aug_test['is_litigated'].values
proba_g  = lgb_gnn.predict_proba(aug_test[FEAT_GNN])[:, 1]
auc_g    = roc_auc_score(y_te, proba_g)
ap_g     = average_precision_score(y_te, proba_g)

tr_proba_g = lgb_gnn.predict_proba(aug_train[FEAT_GNN])[:, 1]
best_thr_g = threshold_metrics(aug_train['is_litigated'].values, tr_proba_g)['threshold']
te_m_g     = threshold_metrics(y_te, proba_g, threshold=best_thr_g)

print(f'\nGNN+LGB  AUC={auc_g:.4f}  AP={ap_g:.4f}')
print(f'  Thr={best_thr_g:.3f}  BA={te_m_g["BA"]:.4f}  F1={te_m_g["F1"]:.4f}  '
      f'F2={te_m_g["F2"]:.4f}  MCC={te_m_g["MCC"]:.4f}  Gmean={te_m_g["Gmean"]:.4f}')

# ── 최종 비교표 ────────────────────────────────────────────────────────────
best_ml = m1_results[best_set]
print('\n' + '='*80)
print(f'{"모델":20s}  {"AUC":6s}  {"AP":6s}  {"BA":6s}  {"F1":6s}  {"F2":6s}  {"MCC":6s}  {"Gmean":6s}')
print('-'*80)
print(f'{"LGB (22 feat)":20s}  {best_ml["test_auc"]:.4f}  {best_ml["test_ap"]:.4f}  '
      f'{best_ml.get("test_BA",float("nan")):.4f}  {best_ml.get("test_F1",float("nan")):.4f}  '
      f'{best_ml.get("test_F2",float("nan")):.4f}  {best_ml.get("test_MCC",float("nan")):.4f}  '
      f'{best_ml.get("test_Gmean",float("nan")):.4f}')
print(f'{"GNN+LGB (86 feat)":20s}  {auc_g:.4f}  {ap_g:.4f}  '
      f'{te_m_g["BA"]:.4f}  {te_m_g["F1"]:.4f}  '
      f'{te_m_g["F2"]:.4f}  {te_m_g["MCC"]:.4f}  {te_m_g["Gmean"]:.4f}')
print('='*80)